# 🍜 STEP 04 — MCP 연동: 카카오 지도 API / 법령정보 API

**Model Context Protocol(MCP)**을 사용하면 외부 API를 표준 툴로
에이전트에 연결할 수 있습니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| MCP 개요 | 에이전트 ↔ 외부 시스템 표준 연결 프로토콜 |
| `McpServerPlugin` (SK) | MCP 서버를 SK 플러그인으로 사용 |
| stdio MCP 서버 | 로컬 프로세스로 실행하는 MCP 서버 패턴 |
| API → MCP 래핑 | 카카오 API, 법령 API를 MCP 툴로 변환 |


---
## MCP 개념 이해

```
SK Agent
  └── McpServerPlugin  ←→  MCP Server (stdio/HTTP)
                              ├── 카카오 지도 API 호출
                              ├── 국가법령정보센터 API 호출
                              └── 소상공인 상권정보 API 호출
```

MCP는 LLM 에이전트와 외부 도구 사이의 **표준 통신 프로토콜**입니다.
- **stdio 방식**: 로컬 프로세스로 MCP 서버 실행 (개발·테스트 용이)
- **HTTP 방식**: REST 엔드포인트로 MCP 서버 운영 (프로덕션)

> **F&B 시스템 연결**: LocationAgent의 카카오 지도 API, LegalTaxAgent의 법령 API를 MCP로 연결합니다.

In [ ]:
# MCP 지원을 위한 추가 패키지
%pip install semantic-kernel[mcp] mcp

In [1]:
import asyncio, os, json
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.functions import KernelArguments

load_dotenv()

def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        api_key=os.getenv('AZURE_OPENAI_API_KEY'),
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview'),
    ))
    return k

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

print('임포트 완료')


임포트 완료


---
## 1. MCP 서버 직접 구현 — 카카오 지도 API 래핑

> 아래 코드를 `kakao_mcp_server.py`로 저장하면 독립적인 MCP 서버가 됩니다.
> SK 에이전트는 이 서버를 `McpServerPlugin`으로 연결합니다.

In [2]:
# ✅ [FIX v5] mcp 1.26.0 정확한 방식
# FastMCP.run(transport='sse') → host/port 미지원
# FastMCP._mcp_server + SseServerTransport + uvicorn 직접 조합
#
# 실행: python seoul_commercial_mcp_server_sse.py
# ============================================================
FASTMCP_SERVER_CODE = '''import os
import json
import urllib.request
import urllib.parse
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

load_dotenv()

mcp = FastMCP("seoul-commercial-server")

SEOUL_API_KEY = os.getenv("SEOUL_API_KEY", "")
BASE_URL = "http://openapi.seoul.go.kr:8088"

DONG_CODE_MAP = {
    "홍대":     "11440680",
    "홍대입구":  "11440680",
    "강남":     "11680640",
    "역삼":     "11680640",
    "신촌":     "11410530",
    "이태원":   "11170640",
    "건대":     "11305710",
    "합정":     "11440620",
    "잠실":     "11710720",
    "종로":     "11110530",
}

INDUSTRY_CODE_MAP = {
    "카페":       "CS100001",
    "한식":       "CS100001",
    "일반음식점":  "CS100001",
    "치킨":       "CS100003",
    "피자":       "CS100004",
    "패스트푸드":  "CS100005",
    "분식":       "CS100006",
}


@mcp.tool()
def get_estimated_sales(location: str, business_type: str, quarter: str = "20243") -> str:
    """
    Query Seoul commercial area estimated monthly sales
    by administrative district and business type (OA-22175 VwsmAdstrdSelngW).
    location: Location in Korean (e.g. 홍대, 강남, 잠실)
    business_type: Business type in Korean (e.g. 카페, 일반음식점)
    quarter: Quarter YYYYQ (e.g. 20243 = 2024 Q3)
    """
    dong_code     = DONG_CODE_MAP.get(location, "")
    industry_code = INDUSTRY_CODE_MAP.get(business_type, "CS100001")

    if not SEOUL_API_KEY:
        return json.dumps({"error": "SEOUL_API_KEY not set"}, ensure_ascii=False)
    if not dong_code:
        return json.dumps({"error": "Unsupported location", "supported": list(DONG_CODE_MAP.keys())}, ensure_ascii=False)

    url = (
        "%s/%s/json/VwsmAdstrdSelngW/1/5"
        "?STDR_YYQU_CD=%s&ADSTRD_CD=%s&SVC_INDUTY_CD=%s"
    ) % (BASE_URL, urllib.parse.quote(SEOUL_API_KEY), quarter, dong_code, industry_code)

    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            data = json.loads(r.read().decode())
        rows = data.get("VwsmAdstrdSelngW", {}).get("row", [])
        if rows:
            row = rows[0]
            result = {
                "location":           location,
                "dong_name":          row.get("ADSTRD_CD_NM", ""),
                "business_type":      row.get("SVC_INDUTY_CD_NM", business_type),
                "quarter":            row.get("STDR_YYQU_CD", quarter),
                "monthly_sales_krw":  int(row.get("THSMON_SELNG_AMT", 0)),
                "monthly_tx_count":   int(row.get("THSMON_SELNG_CO", 0)),
                "weekday_sales_krw":  int(row.get("MDWK_SELNG_AMT", 0)),
                "weekend_sales_krw":  int(row.get("WKEND_SELNG_AMT", 0)),
                "time_11_14_krw":     int(row.get("TMZON_11_14_SELNG_AMT", 0)),
                "time_17_21_krw":     int(row.get("TMZON_17_21_SELNG_AMT", 0)),
                "male_sales_krw":     int(row.get("ML_SELNG_AMT", 0)),
                "female_sales_krw":   int(row.get("FML_SELNG_AMT", 0)),
                "age_20s_krw":        int(row.get("AGRDE_20_SELNG_AMT", 0)),
                "age_30s_krw":        int(row.get("AGRDE_30_SELNG_AMT", 0)),
                "age_40s_krw":        int(row.get("AGRDE_40_SELNG_AMT", 0)),
                "source":             "서울시 상권분석서비스 VwsmAdstrdSelngW (OA-22175)",
            }
        else:
            result = {"message": "No data", "location": location, "quarter": quarter}
    except Exception as e:
        result = {"error": str(e)}

    return json.dumps(result, ensure_ascii=False)


@mcp.tool()
def get_store_count(location: str, business_type: str, quarter: str = "20243") -> str:
    """
    Query store count and open/close rates
    by administrative district and business type (OA-22172 VwsmAdstrdStorW).
    location: Location in Korean (e.g. 홍대, 강남, 잠실)
    business_type: Business type in Korean (e.g. 카페, 일반음식점)
    quarter: Quarter YYYYQ (e.g. 20243 = 2024 Q3)
    """
    dong_code     = DONG_CODE_MAP.get(location, "")
    industry_code = INDUSTRY_CODE_MAP.get(business_type, "CS100001")

    if not SEOUL_API_KEY:
        return json.dumps({"error": "SEOUL_API_KEY not set"}, ensure_ascii=False)
    if not dong_code:
        return json.dumps({"error": "Unsupported location", "supported": list(DONG_CODE_MAP.keys())}, ensure_ascii=False)

    url = (
        "%s/%s/json/VwsmAdstrdStorW/1/5"
        "?STDR_YYQU_CD=%s&ADSTRD_CD=%s&SVC_INDUTY_CD=%s"
    ) % (BASE_URL, urllib.parse.quote(SEOUL_API_KEY), quarter, dong_code, industry_code)

    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            data = json.loads(r.read().decode())
        rows = data.get("VwsmAdstrdStorW", {}).get("row", [])
        if rows:
            row = rows[0]
            result = {
                "location":        location,
                "dong_name":       row.get("ADSTRD_CD_NM", ""),
                "business_type":   row.get("SVC_INDUTY_CD_NM", business_type),
                "quarter":         row.get("STDR_YYQU_CD", quarter),
                "store_count":     int(row.get("STOR_CO", 0)),
                "open_rate_pct":   float(row.get("OPBIZ_RATE", 0)),
                "close_rate_pct":  float(row.get("CLSBIZ_RATE", 0)),
                "source":          "서울시 상권분석서비스 VwsmAdstrdStorW (OA-22172)",
            }
        else:
            result = {"message": "No data", "location": location, "quarter": quarter}
    except Exception as e:
        result = {"error": str(e)}

    return json.dumps(result, ensure_ascii=False)


if __name__ == "__main__":
    import uvicorn
    from mcp.server.sse import SseServerTransport
    from starlette.applications import Starlette
    from starlette.routing import Route, Mount
    from starlette.requests import Request

    port = int(os.getenv("MCP_PORT", "8001"))
    print("Seoul Commercial MCP Server (mcp 1.26 / SSE) on http://localhost:%d/sse" % port)
    print("SEOUL_API_KEY:", "설정됨 ✅" if SEOUL_API_KEY else "❌ 미설정")

    # mcp 1.26.0: SseServerTransport + Starlette 직접 구성
    sse_transport = SseServerTransport("/messages/")

    async def handle_sse(request: Request):
        async with sse_transport.connect_sse(
            request.scope, request.receive, request._send
        ) as streams:
            await mcp._mcp_server.run(
                streams[0], streams[1],
                mcp._mcp_server.create_initialization_options()
            )

    starlette_app = Starlette(routes=[
        Route("/sse", endpoint=handle_sse),
        Mount("/messages/", app=sse_transport.handle_post_message),
    ])

    uvicorn.run(starlette_app, host="0.0.0.0", port=port)
'''

with open('seoul_commercial_mcp_server_sse.py', 'w', encoding='utf-8') as f:
    f.write(FASTMCP_SERVER_CODE)

print('seoul_commercial_mcp_server_sse.py 저장 완료! (mcp 1.26 호환)')
print('터미널: python seoul_commercial_mcp_server_sse.py')


seoul_commercial_mcp_server_sse.py 저장 완료! (mcp 1.26 호환)
터미널: python seoul_commercial_mcp_server_sse.py


---
## 2. SK에서 MCP 서버 연결 — `McpServerPlugin`

In [3]:
# ✅ [FIX v3] MCPStdioPlugin(Windows 불가) → MCPSsePlugin(HTTP/SSE)
# 사전 조건: 터미널에서 seoul_commercial_mcp_server_sse.py 실행 중이어야 함
from semantic_kernel.connectors.mcp import MCPSsePlugin

async def create_location_agent_with_mcp() -> tuple:
    """
    Creates a LocationAgent connected to the Seoul Commercial Area MCP server via SSE.
    Prerequisites: seoul_commercial_mcp_server_sse.py must be running on port 8001.
    """
    kernel = make_kernel()

    # SSE 방식: 외부 HTTP 서버에 연결 (subprocess 불필요)
    mcp_plugin = MCPSsePlugin(
        name='SeoulCommercial',
        description='Seoul commercial area analysis using open data API',
        url='http://localhost:8001/sse',
    )
    await mcp_plugin.connect()
    kernel.add_plugin(mcp_plugin)

    agent = ChatCompletionAgent(
        name='LocationAgent',
        description='Commercial area analysis agent using Seoul Open Data API',
        instructions=(
            'You are a commercial area analysis expert for F&B startups in Seoul. '
            'Use get_estimated_sales to retrieve monthly estimated sales data. '
            'Use get_store_count to retrieve competitor store count and open/close rates. '
            'Always call both tools and synthesize results into a Risk / Opportunity analysis. '
            'Always respond in Korean.'
        ),
        kernel=kernel,
        arguments=KernelArguments(settings=make_auto_settings()),
    )
    return agent, mcp_plugin

print('SSE 방식 LocationAgent 팩토리 함수 정의 완료')
print('⚠️  seoul_commercial_mcp_server_sse.py 가 실행 중이어야 합니다')
print('    터미널: python seoul_commercial_mcp_server_sse.py')


SSE 방식 LocationAgent 팩토리 함수 정의 완료
⚠️  seoul_commercial_mcp_server_sse.py 가 실행 중이어야 합니다
    터미널: python seoul_commercial_mcp_server_sse.py


In [4]:
SEOUL_KEY = os.getenv('SEOUL_API_KEY', '')

if SEOUL_KEY:
    location_agent_mcp, mcp_plugin = await create_location_agent_with_mcp()
    try:
        response = await location_agent_mcp.get_response(
            messages=(
                'Analyze the commercial area for a cafe in Hongdae (홍대). '
                'Get estimated monthly sales and store count data for Q3 2024. '
                'Provide a Risk / Opportunity analysis. Respond in Korean.'
            )
        )
        print(response.content)
    finally:
        await mcp_plugin.close()
else:
    print('SEOUL_API_KEY가 설정되지 않았습니다.')
    print('.env 파일에 SEOUL_API_KEY=발급키 를 추가하세요.')
    print('발급: https://data.seoul.go.kr → 회원가입 → API 키 발급')


홍대 지역의 2024년 3분기 카페 사업에 대한 분석을 아래와 같이 제공합니다.

### 매출 분석
1. **월간 예상 매출**: 23,947,025원
2. **거래 횟수**: 1574건
3. **평일 매출**: 21,587,186원
4. **주말 매출**: 2,359,839원
5. **주요 시간대 매출**: 
   - 오전 11시부터 오후 2시까지: 17,664,188원
   - 오후 5시부터 오후 9시까지: 2,645,537원
6. **성별 매출**:
   - 남성: 10,140,200원
   - 여성: 10,038,004원
7. **연령대별 매출**:
   - 20대: 1,229,751원
   - 30대: 2,073,871원
   - 40대: 5,535,015원

### 경쟁 분석
1. **영업 중인 카페 수**: 18개
2. **신규 오픈율 및 폐업율**: 0% (즉, 해당 분기에 신규 오픈이나 폐업이 없는 상태)

### 리스크 / 기회 분석
**기회 분석**:
- 홍대는 20대와 30대 중심으로 소비층이 형성되어 있으며, 카페에 대한 수요가 꾸준한 지역입니다.
- 폐업률이 0%로 매우 안정적인 시장 환경을 보여주고 있어 새로운 진입을 고민하는 창업자에게 긍정적인 지표입니다.

**리스크 분석**:
- 경쟁 업체가 이미 18개 존재하며, 경쟁이 치열합니다.
- 매출이 특정 시간대와 연령대에 집중되어 있어, 다양한 고객층을 대상으로 매출 다변화 전략이 필요합니다.

결론적으로, 홍대는 카페 사업에 투자할 수 있는 기회가 있는 지역이나, 경쟁이 심한 만큼 차별화된 마케팅과 운영 전략이 필요합니다.


---
## 3. 국가법령정보센터 API MCP 서버

> LegalTaxAgent에 연결할 법령 API MCP 서버 코드입니다.

In [ ]:
LAW_MCP_SERVER_CODE = '''
import asyncio
import os
import httpx
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

server = Server("law-info-server")

LAW_API_KEY = os.getenv("LAW_API_KEY", "")  # 국가법령정보센터 발급 키
LAW_BASE = "https://www.law.go.kr/DRF/lawSearch.do"


@server.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="search_law",
            description="국가법령정보센터에서 법령을 검색합니다. 식품위생법, 상가임대차보호법 등 F&B 관련 법령 조회.",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색할 법령명 또는 키워드"},
                },
                "required": ["query"],
            },
        ),
        Tool(
            name="get_law_article",
            description="특정 법령의 조문 내용을 조회합니다.",
            inputSchema={
                "type": "object",
                "properties": {
                    "law_name": {"type": "string", "description": "법령명 (예: 식품위생법)"},
                    "article_no": {"type": "string", "description": "조문 번호 (예: 제37조)"},
                },
                "required": ["law_name", "article_no"],
            },
        ),
    ]


@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    async with httpx.AsyncClient() as client:
        if name == "search_law":
            resp = await client.get(
                LAW_BASE,
                params={
                    "OC": LAW_API_KEY,
                    "target": "law",
                    "type": "JSON",
                    "query": arguments["query"],
                    "display": 5,
                },
            )
            return [TextContent(type="text", text=resp.text)]

    return [TextContent(type="text", text="알 수 없는 툴: %s" % name)]


async def main():
    async with stdio_server() as (read, write):
        await server.run(read, write, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("law_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(LAW_MCP_SERVER_CODE)

print("law_mcp_server.py 저장 완료!")